# 4.1 Custom GB grid LCA — Carbon Intensity API

Builds a custom GB or regional electricity-mix foreground activity from the National Energy System Operator Carbon Intensity API instead of `df_fuel_ckan.csv`, then runs the foreground H₂-technology LCAs with that mix substituted for direct electricity inputs.

This notebook still uses `dashboard_config.py` as the dashboard:

- `GRID_TIME_MODE`, `GRID_SINGLE_DATETIME`, `GRID_RANGE_START`, `GRID_RANGE_END`, `GRID_YEAR`
- `METHOD_MODE`, `APPLY_LOSSES`, `TECH_SELECTED`, `GRID_OUTPUT_DIR`
- `WIND_LAT` and `WIND_LON`, so the grid scenario is tied to the same coordinates as the wind scenario

The Carbon Intensity API does **not** accept latitude/longitude directly. By default this notebook maps `WIND_LAT` / `WIND_LON` to the nearest Carbon Intensity region and uses the regional endpoint. Set `CARBON_API_SCOPE = "national"` in the first code cell if you want the national GB generation mix instead.

The API generation mix is percentage-only, so the notebook creates a pseudo-`GENERATION = 100` row. This preserves the existing Brightway helper logic, because the LCA calculations use generation shares rather than absolute MW.


In [ ]:
# Run flags / global settings come from dashboard_config.py.
# Ecoinvent queries + candidate indexes for THIS notebook live below.
from dashboard_config import *
import dashboard_config as cfg
import lca_helpers as H

from pathlib import Path
import json
import time
import math
import hashlib
import requests
import numpy as np
import pandas as pd

# =============================================================================
# Carbon Intensity API settings for this 4.1 notebook
# =============================================================================
CARBON_API_BASE = "https://api.carbonintensity.org.uk"

# "regional_auto" reads cfg.WIND_LAT / cfg.WIND_LON and maps them to the nearest
# Carbon Intensity region. "national" ignores coordinates and uses GB national mix.
CARBON_API_SCOPE = "regional_auto"          # "regional_auto" or "national"
CARBON_API_REGION_ID_OVERRIDE = None        # e.g. 13 for London; leave None for auto
CARBON_API_LOCAL_TZ = "Europe/London"       # dashboard datetimes are interpreted as UK local time
CARBON_API_MAX_DAYS_PER_REQUEST = 13        # keep below national intensity 14-day limit
CARBON_API_REQUEST_SLEEP_S = 0.05
CARBON_API_USE_CACHE = True
CARBON_API_CACHE_DIR = Path("carbon_intensity_api_cache")

# National/regional API returns aggregate "imports" only. If lca_helpers expects
# detailed import row keys, use helper weights when available; otherwise split equally.
CARBON_API_DEFAULT_IMPORT_SPLIT = {
    "IMPORTS_FR": 1/3,
    "IMPORTS_IE": 1/3,
    "IMPORTS_NL": 1/3,
}

# API has one aggregate wind percentage. Keep embedded wind separate only if the
# helper explicitly asks for it; default is all wind in WIND and zero WIND_EMB.
CARBON_API_WIND_EMB_SHARE = 0.0

print_dashboard()
print()
print("Carbon Intensity API settings")
print("-----------------------------")
print("Scope:          ", CARBON_API_SCOPE)
print("API base:       ", CARBON_API_BASE)
print("Dashboard coords:", getattr(cfg, "WIND_LAT", None), getattr(cfg, "WIND_LON", None))
print("Time zone:      ", CARBON_API_LOCAL_TZ)
print("Cache dir:      ", CARBON_API_CACHE_DIR.resolve())
print()

ei, bio, fg_db, method = H.setup_brightway()


In [ ]:
if not RUN_GRID_SCENARIO_LCA:
    raise SystemExit(
        "RUN_GRID_SCENARIO_LCA is False in dashboard_config.py. "
        "Set it True and re-run this notebook."
    )

if fg_db is None:
    raise SystemExit(
        "Foreground database not found. Run tech_lca_foreground.ipynb first "
        "(set RUN_BUILD_FOREGROUND_DATABASE=True in dashboard_config.py)."
    )

## Custom GB electricity mix — ecoinvent template

Three dicts that fully describe how the custom GB grid is built:

| dict | what it picks |
|---|---|
| `GRID_CANDIDATE_INDEX` | one ecoinvent process per generation/import component |
| `GRID_INFRASTRUCTURE`  | constant non-energy technosphere inputs per kWh (transmission, SF6) |
| `GRID_EMISSIONS`       | constant non-energy biosphere emissions per kWh (SF6, N2O, ozone) |

Each component lists a `query` (passed to `ei.search(...)` /  `bio.search(...)`)
and a `candidate_index` (which hit to use). Set `SHOW_GRID_ECOINVENT_CANDIDATES`
in [dashboard_config.py](dashboard_config.py) to print every candidate list, then
edit the integers below to switch processes.

In [ ]:
# --- Generation / import components ---------------------------------------
# TECH_SPECS (in lca_helpers.py) supplies the queries; choose the candidate.
GRID_CANDIDATE_INDEX = {
    # Conventional generation
    "GAS":                3,
    "COAL":               0,
    "NUCLEAR":            1,
    # Wind sub-technologies (composite, split by WIND_GROUP_WEIGHTS)
    "WIND_GT3_ONSHORE":   0,
    "WIND_13_OFFSHORE":   0,
    "WIND_13_ONSHORE":    0,
    "WIND_EMB":           0,
    # Hydro
    "HYDRO":              0,
    # Imports (composite, split by IMPORT_GROUP_WEIGHTS)
    "IMPORTS_FR":         0,
    "IMPORTS_IE":         0,
    "IMPORTS_NL":         4,
    # Other generation
    "BIOMASS":            0,
    "OTHER":              0,
    "SOLAR":              0,
    "STORAGE":            0,
}

# --- Constant non-energy technosphere inputs (per kWh) --------------------
GRID_INFRASTRUCTURE = {
    "TRANSMISSION_HV": {
        "query":            "market for transmission network electricity high voltage",
        "name_contains":    ["market for transmission network", "high voltage"],
        "name_excludes":    ["direct current", "maintenance"],
        "candidate_index":  0,
        "amount":           6.58e-9,
        "unit":             "kilometer",
    },
    "SF6_INPUT": {
        "query":            "market for sulfur hexafluoride, liquid",
        "name_contains":    "market for sulfur hexafluoride",
        "candidate_index":  0,
        "amount":           2.99e-9,
        "unit":             "kilogram",
    },
}

# --- Constant non-energy biosphere emissions (per kWh) --------------------
GRID_EMISSIONS = {
    "SF6_AIR": {
        "query":            "Sulfur hexafluoride",
        "candidate_index":  0,
        "categories":       ("air", "non-urban air or from high stacks"),
        "amount":           2.99e-9,
        "unit":             "kilogram",
    },
    "N2O_AIR": {
        "query":            "Dinitrogen monoxide",
        "candidate_index":  0,
        "categories":       ("air", "non-urban air or from high stacks"),
        "amount":           4.90e-8,
        "unit":             "kilogram",
    },
    "OZONE_AIR": {
        "query":            "Ozone",
        "candidate_index":  0,
        "categories":       ("air", "non-urban air or from high stacks"),
        "amount":           4.15e-8,
        "unit":             "kilogram",
    },
}

print("Custom GB electricity mix — candidate indexes")
print("---------------------------------------------")
for tech_key, idx in GRID_CANDIDATE_INDEX.items():
    print(f"  {tech_key:<22} candidate #{idx}")

print()
print("Constant non-energy inputs (per kWh)")
print("------------------------------------")
for k, spec in GRID_INFRASTRUCTURE.items():
    print(f"  infra / {k:<24} {spec['amount']:>10.3e} {spec.get('unit', ''):<10}"
          f" [{spec['query']!r}, candidate #{spec.get('candidate_index', 0)}]")
for k, spec in GRID_EMISSIONS.items():
    if float(spec.get("amount", 0)) <= 0:
        continue
    cats = spec.get("categories") or "(any)"
    print(f"  emis  / {k:<24} {spec['amount']:>10.3e} {spec.get('unit', ''):<10}"
          f" [{spec['query']!r}, categories={cats}]")
# Show candidate lists when this notebook runs (set False to suppress).
SHOW_GRID_ECOINVENT_CANDIDATES = True


## Fetch Carbon Intensity API data and select dashboard timeslices

This replaces `H.load_grid_csv()` / `H.select_grid_rows(df)`. The output is deliberately shaped like the old CSV dataframe: it has `DATETIME`, `CARBON_INTENSITY`, fuel columns, fuel percentage columns, optional representative-day tags, and metadata from the API.


In [ ]:
# =============================================================================
# Carbon Intensity API client + dataframe conversion
# =============================================================================

CI_FUELS = ["gas", "coal", "biomass", "nuclear", "hydro", "imports", "other", "wind", "solar", "storage"]

# Approximate centroids for the 14 detailed GB regions exposed by the API.
# The API accepts region IDs / postcodes, not raw coordinates, so this is only
# a pragmatic coordinate-to-region bridge for dashboard-driven modelling.
CI_REGION_CENTROIDS = {
    1:  {"shortname": "North Scotland",      "lat": 57.7, "lon": -4.0},
    2:  {"shortname": "South Scotland",      "lat": 55.7, "lon": -3.5},
    3:  {"shortname": "North West England",  "lat": 53.6, "lon": -2.7},
    4:  {"shortname": "North East England",  "lat": 54.9, "lon": -1.8},
    5:  {"shortname": "Yorkshire",           "lat": 53.9, "lon": -1.3},
    6:  {"shortname": "North Wales",         "lat": 53.1, "lon": -3.6},
    7:  {"shortname": "South Wales",         "lat": 51.6, "lon": -3.5},
    8:  {"shortname": "West Midlands",       "lat": 52.5, "lon": -2.1},
    9:  {"shortname": "East Midlands",       "lat": 52.9, "lon": -0.8},
    10: {"shortname": "East England",        "lat": 52.2, "lon":  0.5},
    11: {"shortname": "South West England",  "lat": 50.9, "lon": -3.6},
    12: {"shortname": "South England",       "lat": 51.0, "lon": -1.1},
    13: {"shortname": "London",              "lat": 51.5, "lon": -0.1},
    14: {"shortname": "South East England",  "lat": 51.3, "lon":  0.4},
}

SEASON_ORDER_API = ["winter", "spring", "summer", "autumn"]
SEASON_MONTHS_API = {
    "winter": [12, 1, 2],
    "spring": [3, 4, 5],
    "summer": [6, 7, 8],
    "autumn": [9, 10, 11],
}


def _haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lam = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lam / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def infer_ci_region_from_dashboard_coords():
    """Map dashboard wind coordinates to the nearest Carbon Intensity region ID."""
    if CARBON_API_REGION_ID_OVERRIDE is not None:
        region_id = int(CARBON_API_REGION_ID_OVERRIDE)
        meta = CI_REGION_CENTROIDS.get(region_id, {"shortname": f"Region {region_id}"})
        return region_id, meta.get("shortname", f"Region {region_id}"), 0.0

    lat = float(getattr(cfg, "WIND_LAT"))
    lon = float(getattr(cfg, "WIND_LON"))
    ranked = []
    for region_id, meta in CI_REGION_CENTROIDS.items():
        dist = _haversine_km(lat, lon, meta["lat"], meta["lon"])
        ranked.append((dist, region_id, meta["shortname"]))
    dist, region_id, shortname = sorted(ranked)[0]
    return region_id, shortname, dist


def _localise_dashboard_time(value):
    """Interpret dashboard datetimes as UK local time, then return tz-aware Timestamp."""
    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        try:
            ts = ts.tz_localize(CARBON_API_LOCAL_TZ, nonexistent="shift_forward")
        except Exception:
            # Ambiguous autumn-clock-change half-hours: choose the first occurrence.
            ts = ts.tz_localize(CARBON_API_LOCAL_TZ, ambiguous=True, nonexistent="shift_forward")
    else:
        ts = ts.tz_convert(CARBON_API_LOCAL_TZ)
    return ts


def _api_z(value):
    return _localise_dashboard_time(value).tz_convert("UTC").strftime("%Y-%m-%dT%H:%MZ")


def _cache_key(endpoint):
    digest = hashlib.sha256(endpoint.encode("utf-8")).hexdigest()[:24]
    safe = endpoint.strip("/").replace("/", "__").replace(":", "-")
    return CARBON_API_CACHE_DIR / f"{digest}_{safe}.json"


def carbon_api_get(endpoint):
    """GET an API endpoint, with lightweight local JSON caching."""
    CARBON_API_CACHE_DIR.mkdir(exist_ok=True)
    cache_path = _cache_key(endpoint)
    if CARBON_API_USE_CACHE and cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as f:
            return json.load(f)

    url = CARBON_API_BASE.rstrip("/") + endpoint
    response = requests.get(url, headers={"Accept": "application/json"}, timeout=45)
    if response.status_code != 200:
        raise RuntimeError(f"Carbon Intensity API returned HTTP {response.status_code}: {response.text[:500]}")
    payload = response.json()
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(f"Carbon Intensity API error: {payload['error']}")

    if CARBON_API_USE_CACHE:
        with cache_path.open("w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
    time.sleep(float(CARBON_API_REQUEST_SLEEP_S))
    return payload


def _iter_chunks_local(start, end_inclusive, max_days=None):
    """Yield local-time chunks. End is inclusive at half-hour resolution."""
    max_days = int(max_days or CARBON_API_MAX_DAYS_PER_REQUEST)
    start_ts = _localise_dashboard_time(start)
    end_ts = _localise_dashboard_time(end_inclusive)
    request_end = end_ts + pd.Timedelta(minutes=30)

    current = start_ts
    step = pd.Timedelta(days=max_days)
    while current < request_end:
        nxt = min(current + step, request_end)
        yield current, nxt
        current = nxt


def _flatten_generation_records(records, source_scope, region_id=None, region_name=None):
    rows = []
    for rec in records:
        row = {}
        from_utc = pd.to_datetime(rec.get("from"), utc=True)
        to_utc = pd.to_datetime(rec.get("to"), utc=True)
        from_local = from_utc.tz_convert(CARBON_API_LOCAL_TZ)
        row["DATETIME"] = from_local.tz_localize(None)
        row["API_FROM_UTC"] = from_utc.isoformat()
        row["API_TO_UTC"] = to_utc.isoformat()
        row["api_scope"] = source_scope
        row["api_region_id"] = region_id
        row["api_region_name"] = region_name
        row["wind_lat"] = float(getattr(cfg, "WIND_LAT", np.nan))
        row["wind_lon"] = float(getattr(cfg, "WIND_LON", np.nan))

        for fuel in CI_FUELS:
            row[f"api_{fuel}"] = 0.0
        for item in rec.get("generationmix", []) or []:
            fuel = str(item.get("fuel", "")).strip().lower()
            if fuel:
                row[f"api_{fuel}"] = float(item.get("perc", 0.0) or 0.0)

        intensity = rec.get("intensity") or {}
        row["carbon_intensity_forecast"] = intensity.get("forecast", np.nan)
        row["carbon_intensity_actual"] = intensity.get("actual", np.nan)
        row["carbon_intensity_index"] = intensity.get("index", None)
        actual = row["carbon_intensity_actual"]
        forecast = row["carbon_intensity_forecast"]
        row["CARBON_INTENSITY"] = actual if pd.notna(actual) else forecast
        rows.append(row)
    return pd.DataFrame(rows)


def _flatten_national_intensity_records(records):
    rows = []
    for rec in records:
        from_utc = pd.to_datetime(rec.get("from"), utc=True)
        intensity = rec.get("intensity") or {}
        actual = intensity.get("actual", np.nan)
        forecast = intensity.get("forecast", np.nan)
        rows.append({
            "DATETIME": from_utc.tz_convert(CARBON_API_LOCAL_TZ).tz_localize(None),
            "carbon_intensity_forecast": forecast,
            "carbon_intensity_actual": actual,
            "carbon_intensity_index": intensity.get("index", None),
            "CARBON_INTENSITY": actual if pd.notna(actual) else forecast,
        })
    return pd.DataFrame(rows)


def _regional_records_from_payload(payload, expected_region_id, expected_region_name):
    data = payload.get("data", []) if isinstance(payload, dict) else []
    if not data:
        return []

    # /regional/intensity/{from}/{to}/regionid/{regionid} usually returns:
    # [{regionid, dnoregion, shortname, data: [{from, to, intensity, generationmix}, ...]}]
    first = data[0]
    if isinstance(first, dict) and isinstance(first.get("data"), list):
        region_name = first.get("shortname") or expected_region_name
        region_id = first.get("regionid") or expected_region_id
        return first["data"], region_id, region_name

    # Be defensive: /regional/intensity/{from}/{to} returns timeslices with regions.
    records = []
    region_name = expected_region_name
    region_id = expected_region_id
    for timeslice in data:
        for region in timeslice.get("regions", []) or []:
            if int(region.get("regionid", -1)) == int(expected_region_id):
                rec = {
                    "from": timeslice.get("from"),
                    "to": timeslice.get("to"),
                    "intensity": region.get("intensity"),
                    "generationmix": region.get("generationmix"),
                }
                records.append(rec)
                region_name = region.get("shortname") or region_name
                region_id = region.get("regionid") or region_id
                break
    return records, region_id, region_name


def fetch_carbon_api_dataframe(start, end_inclusive, scope=None):
    """Fetch Carbon Intensity generation mix rows and return old-CSV-like dataframe."""
    scope = scope or CARBON_API_SCOPE
    all_frames = []

    if scope == "regional_auto":
        region_id, region_name, region_dist_km = infer_ci_region_from_dashboard_coords()
        print(f"Using regional API: region {region_id} — {region_name} ")
        print(f"Mapped from WIND_LAT/WIND_LON; centroid distance ≈ {region_dist_km:.1f} km")
        for chunk_start, chunk_end in _iter_chunks_local(start, end_inclusive):
            endpoint = f"/regional/intensity/{_api_z(chunk_start)}/{_api_z(chunk_end)}/regionid/{region_id}"
            payload = carbon_api_get(endpoint)
            records, rid, rname = _regional_records_from_payload(payload, region_id, region_name)
            if records:
                all_frames.append(_flatten_generation_records(records, "regional_auto", rid, rname))

    elif scope == "national":
        print("Using national GB generation mix API")
        gen_frames = []
        int_frames = []
        for chunk_start, chunk_end in _iter_chunks_local(start, end_inclusive):
            gen_payload = carbon_api_get(f"/generation/{_api_z(chunk_start)}/{_api_z(chunk_end)}")
            gen_records = gen_payload.get("data", []) if isinstance(gen_payload, dict) else []
            if gen_records:
                gen_frames.append(_flatten_generation_records(gen_records, "national", None, "Great Britain"))

            # National generation endpoint does not include intensity, so merge it in.
            int_payload = carbon_api_get(f"/intensity/{_api_z(chunk_start)}/{_api_z(chunk_end)}")
            int_records = int_payload.get("data", []) if isinstance(int_payload, dict) else []
            if int_records:
                int_frames.append(_flatten_national_intensity_records(int_records))

        if gen_frames:
            gen_df = pd.concat(gen_frames, ignore_index=True)
            if int_frames:
                int_df = pd.concat(int_frames, ignore_index=True).drop_duplicates("DATETIME")
                gen_df = gen_df.drop(columns=[
                    "carbon_intensity_forecast", "carbon_intensity_actual", "carbon_intensity_index", "CARBON_INTENSITY"
                ], errors="ignore").merge(int_df, on="DATETIME", how="left")
            all_frames.append(gen_df)
    else:
        raise ValueError("CARBON_API_SCOPE must be 'regional_auto' or 'national'.")

    if not all_frames:
        raise RuntimeError("No records returned from Carbon Intensity API.")

    df_api = pd.concat(all_frames, ignore_index=True)
    df_api = df_api.drop_duplicates("DATETIME").sort_values("DATETIME").reset_index(drop=True)

    start_ts = _localise_dashboard_time(start).tz_localize(None)
    end_ts = _localise_dashboard_time(end_inclusive).tz_localize(None)
    df_api = df_api[(df_api["DATETIME"] >= start_ts) & (df_api["DATETIME"] <= end_ts)].copy()
    if df_api.empty:
        raise RuntimeError("API returned records, but none matched the requested dashboard window.")

    return map_carbon_api_to_lca_schema(df_api)


def _normalise_weight_dict(weights, wanted_keys):
    """Return a clean weight dict for wanted_keys."""
    out = {}
    if isinstance(weights, dict):
        for k in wanted_keys:
            if k in weights:
                out[k] = float(weights[k])
    if not out:
        out = {k: 1.0 for k in wanted_keys}
    total = sum(v for v in out.values() if v > 0)
    if total <= 0:
        return {k: 1.0 / len(wanted_keys) for k in wanted_keys}
    return {k: out.get(k, 0.0) / total for k in wanted_keys}


def _helper_expected_row_keys():
    expected = set()
    if hasattr(H, "FUEL_COLS"):
        expected.update(list(H.FUEL_COLS))
    if hasattr(H, "TECH_SPECS"):
        for spec in H.TECH_SPECS.values():
            rk = spec.get("row_key")
            if rk:
                expected.add(rk)
    return expected


def map_carbon_api_to_lca_schema(df_api):
    """Create the fuel columns expected by lca_helpers from API percentages."""
    df = df_api.copy()

    # Pseudo-MW basis: generation mix is percentage-only, so use total=100.
    df["GENERATION"] = 100.0
    df["api_total_perc"] = df[[f"api_{fuel}" for fuel in CI_FUELS if f"api_{fuel}" in df]].sum(axis=1)

    base_map = {
        "GAS": "api_gas",
        "COAL": "api_coal",
        "BIOMASS": "api_biomass",
        "NUCLEAR": "api_nuclear",
        "HYDRO": "api_hydro",
        "IMPORTS": "api_imports",
        "OTHER": "api_other",
        "SOLAR": "api_solar",
        "STORAGE": "api_storage",
    }

    # Aggregate wind. If you later want embedded wind separated, set
    # CARBON_API_WIND_EMB_SHARE > 0.
    wind_total = df.get("api_wind", 0.0).astype(float)
    emb_share = float(CARBON_API_WIND_EMB_SHARE)
    df["WIND_EMB"] = wind_total * emb_share
    df["WIND_EMB_perc"] = wind_total * emb_share
    df["WIND"] = wind_total * (1.0 - emb_share)
    df["WIND_perc"] = wind_total * (1.0 - emb_share)

    for col, api_col in base_map.items():
        values = df.get(api_col, 0.0).astype(float) if api_col in df else 0.0
        df[col] = values
        df[f"{col}_perc"] = values

    expected = _helper_expected_row_keys()

    # If helper expects detailed import row keys, split aggregate imports.
    import_keys = [k for k in expected if k.startswith("IMPORTS_")]
    if import_keys:
        helper_weights = getattr(H, "IMPORT_GROUP_WEIGHTS", CARBON_API_DEFAULT_IMPORT_SPLIT)
        weights = _normalise_weight_dict(helper_weights, import_keys)
        for k in import_keys:
            df[k] = df["IMPORTS"] * weights.get(k, 0.0)
            df[f"{k}_perc"] = df["IMPORTS_perc"] * weights.get(k, 0.0)

    # If helper expects detailed wind row keys rather than aggregate WIND, split aggregate wind.
    detailed_wind_keys = [k for k in expected if k.startswith("WIND_") and k != "WIND_EMB"]
    if detailed_wind_keys:
        helper_weights = getattr(H, "WIND_GROUP_WEIGHTS", None)
        weights = _normalise_weight_dict(helper_weights, detailed_wind_keys)
        for k in detailed_wind_keys:
            df[k] = df["WIND"] * weights.get(k, 0.0)
            df[f"{k}_perc"] = df["WIND_perc"] * weights.get(k, 0.0)

    # Any missing helper-required fuel columns are validly zero.
    for col in expected:
        if col not in df:
            df[col] = 0.0
        if f"{col}_perc" not in df:
            df[f"{col}_perc"] = 0.0

    # Convenience date fields.
    dt = pd.to_datetime(df["DATETIME"])
    df["DATE"] = dt.dt.date
    df["YEAR"] = dt.dt.year
    df["MONTH"] = dt.dt.month
    df["HH"] = dt.dt.hour * 2 + (dt.dt.minute // 30) + 1

    return df


def _season_for_month(month):
    for season, months in SEASON_MONTHS_API.items():
        if int(month) in months:
            return season
    raise ValueError(f"No season for month {month}")


def seasonal_wind_summary_api(df_in, year, decile=0.10):
    """Representative wind-day summary using API wind percentages."""
    d = df_in.copy()
    d["DATETIME"] = pd.to_datetime(d["DATETIME"])
    d = d[d["DATETIME"].dt.year == int(year)].copy()
    if d.empty:
        raise ValueError(f"No rows for GRID_YEAR={year} in API dataframe.")

    wind_cols = [c for c in ["WIND", "WIND_EMB"] if c in d.columns]
    d["wind_pct"] = d[wind_cols].sum(axis=1)
    d["date"] = d["DATETIME"].dt.date
    daily = d.groupby("date", as_index=False).agg(
        wind_pct=("wind_pct", "mean"),
        month=("MONTH", "first"),
        n_halfhours=("wind_pct", "count"),
    )
    daily["season"] = daily["month"].map(_season_for_month)
    daily["date_ts"] = pd.to_datetime(daily["date"])

    summary = {}
    rep_days = []
    for season in SEASON_ORDER_API:
        s = daily[daily["season"] == season].copy()
        if s.empty:
            continue
        season_avg = float(s["wind_pct"].mean())
        avg_row = s.iloc[(s["wind_pct"] - season_avg).abs().argsort().iloc[0]]

        n_tail = max(1, int(math.ceil(len(s) * float(decile))))
        top = s.sort_values("wind_pct", ascending=False).head(n_tail)
        bottom = s.sort_values("wind_pct", ascending=True).head(n_tail)
        top_target = float(top["wind_pct"].mean())
        bottom_target = float(bottom["wind_pct"].mean())
        top_row = top.iloc[(top["wind_pct"] - top_target).abs().argsort().iloc[0]]
        bottom_row = bottom.iloc[(bottom["wind_pct"] - bottom_target).abs().argsort().iloc[0]]

        summary[season] = {
            "n_days": int(len(s)),
            "season_avg_pct": season_avg,
            "rep_day": pd.Timestamp(avg_row["date_ts"]),
            "rep_day_pct": float(avg_row["wind_pct"]),
            "top10_n": int(len(top)),
            "top10_avg_pct": top_target,
            "top10_rep_day": pd.Timestamp(top_row["date_ts"]),
            "top10_rep_day_pct": float(top_row["wind_pct"]),
            "bottom10_n": int(len(bottom)),
            "bottom10_avg_pct": bottom_target,
            "bottom10_rep_day": pd.Timestamp(bottom_row["date_ts"]),
            "bottom10_rep_day_pct": float(bottom_row["wind_pct"]),
        }
        rep_days.extend([
            {"season": season, "kind": "average", "date": pd.Timestamp(avg_row["date_ts"]), "target_pct": season_avg},
            {"season": season, "kind": "top10", "date": pd.Timestamp(top_row["date_ts"]), "target_pct": top_target},
            {"season": season, "kind": "bottom10", "date": pd.Timestamp(bottom_row["date_ts"]), "target_pct": bottom_target},
        ])
    return summary, daily, rep_days


def collapse_api_day_to_average_row(day_rows, season=None, kind=None, target_pct=np.nan):
    """Collapse one representative day to a single simple daily-average mix row."""
    if day_rows.empty:
        raise ValueError("Cannot collapse empty day_rows.")
    out = {}
    numeric_cols = [c for c in day_rows.columns if pd.api.types.is_numeric_dtype(day_rows[c])]
    for c in numeric_cols:
        out[c] = float(day_rows[c].mean())
    first = day_rows.iloc[0]
    date = pd.to_datetime(first["DATETIME"]).normalize()
    out["DATETIME"] = date + pd.Timedelta(hours=12)
    out["DATE"] = date.date()
    out["YEAR"] = int(date.year)
    out["MONTH"] = int(date.month)
    out["HH"] = 25
    out["REP_SEASON"] = season
    out["REP_KIND"] = kind
    out["REP_DATE"] = str(date.date())
    out["REP_TARGET_WIND_PCT"] = target_pct
    out["API_ROW_COUNT"] = int(len(day_rows))

    # Keep stable text metadata.
    for c in ["api_scope", "api_region_id", "api_region_name", "carbon_intensity_index"]:
        if c in day_rows.columns:
            out[c] = first.get(c)
    return out


def select_carbon_api_rows_from_dashboard():
    """Fetch enough API data for the dashboard mode, then select/collapse rows."""
    if GRID_TIME_MODE == "single":
        start = _localise_dashboard_time(GRID_SINGLE_DATETIME)
        end = start
        raw = fetch_carbon_api_dataframe(start, end)
        run_rows = raw.head(1).copy()
        target_label = str(pd.Timestamp(run_rows.iloc[0]["DATETIME"])).replace(" ", "T")

    elif GRID_TIME_MODE == "range":
        raw = fetch_carbon_api_dataframe(GRID_RANGE_START, GRID_RANGE_END)
        run_rows = raw.copy()
        target_label = f"{GRID_RANGE_START} -> {GRID_RANGE_END}"

    elif GRID_TIME_MODE == "year_average":
        year = int(GRID_YEAR)
        start = f"{year}-01-01 00:00:00"
        end = f"{year}-12-31 23:30:00"
        raw = fetch_carbon_api_dataframe(start, end)
        summary, daily, rep_days = seasonal_wind_summary_api(raw, year, decile=float(GRID_REP_DECILE))

        rows = []
        raw["date"] = pd.to_datetime(raw["DATETIME"]).dt.date
        for rep in rep_days:
            day_rows = raw[raw["date"] == rep["date"].date()].copy()
            rows.append(collapse_api_day_to_average_row(
                day_rows,
                season=rep["season"],
                kind=rep["kind"],
                target_pct=rep["target_pct"],
            ))
        run_rows = pd.DataFrame(rows)
        target_label = f"{year}_api_rep_wind_days"
    else:
        raise ValueError("GRID_TIME_MODE must be 'single', 'range' or 'year_average'.")

    return raw, run_rows.reset_index(drop=True), target_label


# Fetch and select rows from the API using the dashboard settings.
df, run_rows, target_label = select_carbon_api_rows_from_dashboard()
row = run_rows.iloc[0]
TARGET_LABEL = str(row["DATETIME"]).replace(" ", "T")

print("Grid scenario selection — Carbon Intensity API")
print("----------------------------------------------")
print("Mode:", GRID_TIME_MODE, "| rows:", len(run_rows), "| window:", target_label)
print("Reference timeslice for candidate checks:", row["DATETIME"])
print("API scope:", row.get("api_scope", CARBON_API_SCOPE))
print("API region:", row.get("api_region_id", "n/a"), row.get("api_region_name", "n/a"))
print("Wind coordinates:", row.get("wind_lat", getattr(cfg, "WIND_LAT", None)), row.get("wind_lon", getattr(cfg, "WIND_LON", None)))
print("Carbon intensity:", row.get("CARBON_INTENSITY", "n/a"), "g CO2/kWh")
print("API mix total:", row.get("api_total_perc", "n/a"), "%")
print()
print(f"{'Technology':<15} {'pseudo-MW':>10}   {'Share %':>8}")
print("-" * 42)
for col in getattr(H, "FUEL_COLS", []):
    mw  = row.get(col, 0)
    pct = row.get(col + "_perc", 0)
    flag = "  <- negative/storage charging" if isinstance(pct, (int, float, np.number)) and pct < 0 else ""
    try:
        print(f"{col:<15} {float(mw):>10.2f}   {float(pct):>7.2f}%{flag}")
    except Exception:
        pass

# A compact view of the raw API fuel fields for checking.
api_cols = [c for c in [f"api_{f}" for f in CI_FUELS] if c in run_rows.columns]
display_cols = ["DATETIME", "CARBON_INTENSITY", "api_scope", "api_region_id", "api_region_name"] + api_cols
run_rows[display_cols].head()


## Inspect ecoinvent candidates (optional)

Set `SHOW_GRID_ECOINVENT_CANDIDATES = True` at the top of this notebook (cell 4)
to print every candidate list (energy components + infrastructure + emissions).
Edit the `candidate_index` integers in the template cell above and re-run.


In [ ]:
if SHOW_GRID_ECOINVENT_CANDIDATES:
    # Energy components (TECH_SPECS supplies the queries; GRID_CANDIDATE_INDEX picks the candidate)
    print(f"\n{'═'*70}\n  ENERGY COMPONENTS — ecoinvent candidates\n{'═'*70}")
    for tech, spec in H.TECH_SPECS.items():
        row_key = spec["row_key"]
        shares = run_rows[row_key + "_perc"] if (row_key + "_perc") in run_rows.columns else None
        max_share = float(shares.max()) if shares is not None else 0.0
        idx = int(GRID_CANDIDATE_INDEX.get(tech, 0))
        if max_share <= 0:
            print(f"\n{'━'*70}\n  {tech}  —  max share in window: {max_share:.2f}%  (SKIPPED)")
            continue
        print(f"\n{'━'*70}")
        print(f"  {tech}  —  max share: {max_share:.2f}%  (picked: candidate #{idx})")
        print(f"  Query: {spec['query']!r}\n")
        pool = list(ei.search(spec["query"]))[:100]
        for i, act in enumerate(pool):
            marker = "  ←" if i == idx else ""
            print(f"  {i:>2} | {act.get('name')} | ref: {act.get('reference product')}"
                  f" | unit: {act.get('unit')} | loc: {act.get('location')}{marker}")
        if idx >= len(pool):
            print(f"  ⚠ candidate_index {idx} out of range ({len(pool)})")

    # Constant non-energy infrastructure
    print(f"\n{'═'*70}\n  NON-ENERGY INFRASTRUCTURE — technosphere candidates\n{'═'*70}")
    for infra_key, spec in GRID_INFRASTRUCTURE.items():
        idx = int(spec.get("candidate_index", 0))
        amt = float(spec.get("amount", 0) or 0)
        print(f"\n{'━'*70}")
        print(f"  {infra_key}  —  amount per kWh: {amt:.3e} {spec.get('unit', '')}"
              f"  (picked: candidate #{idx})")
        print(f"  Query: {spec['query']!r}")
        raw_pool = list(ei.search(spec["query"]))[: int(spec.get("max_results", 100))]
        pool = H.filter_pool(raw_pool, spec)
        if not pool:
            print(f"  ⚠ No candidates after filters (raw hits: {len(raw_pool)})")
            continue
        for i, act in enumerate(pool):
            marker = "  ←" if i == idx else ""
            print(f"  {i:>2} | {act.get('name')} | ref: {act.get('reference product')}"
                  f" | unit: {act.get('unit')} | loc: {act.get('location')}{marker}")

    # Constant non-energy biosphere emissions
    print(f"\n{'═'*70}\n  NON-ENERGY EMISSIONS — biosphere candidates\n{'═'*70}")
    for emis_key, spec in GRID_EMISSIONS.items():
        idx = int(spec.get("candidate_index", 0))
        amt = float(spec.get("amount", 0) or 0)
        cats = spec.get("categories")
        print(f"\n{'━'*70}")
        print(f"  {emis_key}  —  amount per kWh: {amt:.3e} {spec.get('unit', '')}"
              f"  (picked: candidate #{idx})")
        print(f"  Query: {spec['query']!r}   categories filter: {cats}")
        raw_pool = list(bio.search(spec["query"]))[: int(spec.get("max_results", 100))]
        pool = H.filter_biosphere(raw_pool, spec)
        if not pool:
            print(f"  ⚠ No biosphere flows after filter (raw hits: {len(raw_pool)})")
            continue
        for i, flow in enumerate(pool):
            marker = "  ←" if i == idx else ""
            print(f"  {i:>2} | {flow.get('name')} | unit: {flow.get('unit')}"
                  f" | categories: {flow.get('categories')}{marker}")
else:
    print("SHOW_GRID_ECOINVENT_CANDIDATES is False — skipping candidate dump.")

# Resolve the ecoinvent activities for the reference timeslice
selected_processes = H.select_candidate_activities(row, GRID_CANDIDATE_INDEX)


## Build the custom GB electricity activity for the reference timeslice

In [ ]:
new_act = H.build_custom_electricity_activity(
    row, selected_processes, fg_db,
    infrastructure=GRID_INFRASTRUCTURE, emissions=GRID_EMISSIONS,
)
print("Custom electricity activity:", new_act)

## Resolve the selected H2 technologies

In [ ]:
print("Selected technologies for LCA:", SELECTED_LCA_TECHS)
tech_activities = {}
for label in SELECTED_LCA_TECHS:
    code = H.H2_CODES.get(label)
    if code is None:
        print(f"  ⚠ Unknown technology label: {label}")
        continue
    try:
        import bw2data as bd
        tech_activities[label] = bd.get_activity((FOREGROUND_DB, code))
        print(f"  Found: {label}")
    except Exception:
        print(f"  ⚠  Not found: {label} (code={code!r})")

if not tech_activities:
    raise ValueError("No selected foreground activities were found. Check TECH_SELECTED.")

## Batch runner across all selected timeslices

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd

n_total = len(run_rows)
range_records = []

# Preserve API / coordinate metadata in the exported LCA table.
def add_api_metadata(record, csv_row):
    metadata_cols = [
        "api_scope", "api_region_id", "api_region_name",
        "wind_lat", "wind_lon", "api_total_perc",
        "carbon_intensity_forecast", "carbon_intensity_actual", "carbon_intensity_index",
        "REP_SEASON", "REP_KIND", "REP_DATE", "REP_TARGET_WIND_PCT", "API_ROW_COUNT",
    ]
    fuel_cols = [f"api_{fuel}" for fuel in globals().get("CI_FUELS", [])]
    lca_share_cols = [f"{col}_perc" for col in getattr(H, "FUEL_COLS", [])]
    for col in metadata_cols + fuel_cols + lca_share_cols:
        if col in csv_row:
            record[col] = csv_row[col]
    return record

# Initialize cheap-method variables
source_lca_scores = {}
tech_decomposition = {}

if METHOD_MODE == "cheap":
    # Pre-calculate source LCA scores and technology decomposition for cheap method
    _gen = H.select_all_candidate_activities(GRID_CANDIDATE_INDEX)
    source_lca_scores = H.calculate_source_lca_scores(_gen, method)
    tech_decomposition = H.decompose_technology_scores(tech_activities, method, fg_db)

if METHOD_MODE == "exact":
    print(f"Running EXACT batch for {n_total} timeslice(s): {target_label}")
    print("Technologies:", list(tech_activities.keys()), "\n")
    for i, (_, csv_row) in enumerate(run_rows.iterrows()):
        ts = csv_row["DATETIME"]
        print(f"  [{i+1:>2}/{n_total}] {ts}", end="  ", flush=True)
        tmp_elec = None
        try:
            sel = H.select_candidate_activities(csv_row, GRID_CANDIDATE_INDEX)
            tmp_elec = H.build_custom_electricity_activity(
                csv_row, sel, fg_db,
                infrastructure=GRID_INFRASTRUCTURE, emissions=GRID_EMISSIONS,
                verbose=False,
            )
            record = {"datetime": ts, "carbon_intensity": csv_row.get("CARBON_INTENSITY", np.nan)}
            record = add_api_metadata(record, csv_row)
            patch = {}
            for tech_label, tech_act in tech_activities.items():
                score, n_sub = H.run_lca_with_custom_elec(tech_act, tmp_elec, method)
                record[tech_label] = score
                patch[tech_label] = n_sub
            range_records.append(record)
            print("OK (" + ", ".join(f"{k}: {v} elec patched" for k, v in patch.items()) + ")")
        except Exception as e:
            print(f"SKIPPED ({type(e).__name__}: {e})")
        finally:
            if tmp_elec is not None:
                try: tmp_elec.delete()
                except Exception: pass
else:
    print(f"Running CHEAP batch for {n_total} timeslice(s): {target_label}")
    print("Technologies:", list(tech_decomposition.keys()), "\n")
    for i, (_, csv_row) in enumerate(run_rows.iterrows()):
        ts = csv_row["DATETIME"]
        elec_score, elec_amount_sum = H.custom_electricity_score_for_row(csv_row, source_lca_scores)
        record = {
            "datetime": ts,
            "carbon_intensity": csv_row.get("CARBON_INTENSITY", np.nan),
            "custom_electricity_score": elec_score,
            "custom_electricity_input_kwh_per_kwh": elec_amount_sum,
        }
        record = add_api_metadata(record, csv_row)
        for tech_label, parts in tech_decomposition.items():
            record[tech_label] = parts["non_electricity_score"] + parts["electricity_kwh"] * elec_score
        range_records.append(record)
        if (i + 1) % 25 == 0 or i == 0 or i == n_total - 1:
            print(f"  [{i+1:>4}/{n_total}] {ts}  elec={elec_score:.6f} kg CO2eq/kWh")

if not range_records:
    raise RuntimeError("No timeslices were processed.")

range_df = pd.DataFrame(range_records).set_index("datetime")
print(f"\nCompleted {len(range_df)}/{n_total} timeslices ({METHOD_MODE} method).")
range_df.head()

## Optional validation: cheap vs exact

In [ ]:
import pandas as pd

if VALIDATE_CHEAP_METHOD:
    if not source_lca_scores:
        _gen = H.select_all_candidate_activities(GRID_CANDIDATE_INDEX)
        source_lca_scores = H.calculate_source_lca_scores(_gen, method)
    if not tech_decomposition:
        tech_decomposition = H.decompose_technology_scores(tech_activities, method, fg_db)

    validation_records = []
    for _, csv_row in run_rows.head(VALIDATION_N).iterrows():
        sel = H.select_candidate_activities(csv_row, GRID_CANDIDATE_INDEX)
        tmp_elec = H.build_custom_electricity_activity(
            csv_row, sel, fg_db,
            infrastructure=GRID_INFRASTRUCTURE, emissions=GRID_EMISSIONS,
            verbose=False,
        )
        try:
            elec_score, _ = H.custom_electricity_score_for_row(csv_row, source_lca_scores)
            for tech_label, tech_act in tech_activities.items():
                exact_score, n_sub = H.run_lca_with_custom_elec(tech_act, tmp_elec, method)
                parts = tech_decomposition[tech_label]
                cheap_score = parts["non_electricity_score"] + parts["electricity_kwh"] * elec_score
                validation_records.append({
                    "datetime": csv_row["DATETIME"], "technology": tech_label,
                    "exact_score": exact_score, "cheap_score": cheap_score,
                    "difference": cheap_score - exact_score,
                    "pct_difference": 100*(cheap_score-exact_score)/exact_score if exact_score else float("nan"),
                    "electricity_exchanges_replaced": n_sub,
                })
        finally:
            try: tmp_elec.delete()
            except Exception: pass

    validation_df = pd.DataFrame(validation_records)
    print(validation_df.to_string(index=False))
else:
    print("Validation skipped. Set VALIDATE_CHEAP_METHOD=True in dashboard_config.py.")

## Export results CSV (consumed by the dashboard notebook)

In [ ]:
from pathlib import Path

output_dir = Path(GRID_OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)
safe_window = str(target_label).replace(" ", "T").replace(":", "-").replace("->", "_to_")
scope_tag = str(run_rows.iloc[0].get("api_scope", CARBON_API_SCOPE))
region_tag = str(run_rows.iloc[0].get("api_region_id", "national"))
csv_out = output_dir / f"custom_grid_lca_carbon_api_{scope_tag}_r{region_tag}_{METHOD_MODE}_{safe_window}.csv"
range_df.to_csv(csv_out)
print("Saved Carbon-Intensity-API custom-grid LCA results to:")
print(csv_out.resolve())


## Seasonal wind-contribution analysis from Carbon Intensity API data

For `GRID_TIME_MODE = "year_average"`, the LCA rows above use the same representative-day logic shown here: per season, the closest day to average wind, top-decile wind, and bottom-decile wind. Because the API generation mix is percentage-only, this is a simple half-hourly average of wind percentage, not an energy-weighted average by MW.


In [ ]:
import numpy as np
import pandas as pd

ANALYSIS_YEAR = int(getattr(cfg, "GRID_YEAR", 2023))
DECILE = float(getattr(cfg, "GRID_REP_DECILE", 0.10))

# Reuse the API dataframe already fetched. If the current dashboard mode did not
# fetch the whole analysis year, this cell fetches the full year now.
try:
    df
except NameError:
    df, run_rows, target_label = select_carbon_api_rows_from_dashboard()

if not ((pd.to_datetime(df["DATETIME"]).dt.year == ANALYSIS_YEAR).all() and len(df) > 300):
    print(f"Fetching full {ANALYSIS_YEAR} API dataset for seasonal analysis...")
    df_analysis = fetch_carbon_api_dataframe(
        f"{ANALYSIS_YEAR}-01-01 00:00:00",
        f"{ANALYSIS_YEAR}-12-31 23:30:00",
    )
else:
    df_analysis = df.copy()

season_wind_summary, daily, rep_days = seasonal_wind_summary_api(
    df_analysis, ANALYSIS_YEAR, decile=DECILE
)

print(f"Seasonal wind-contribution analysis for {ANALYSIS_YEAR}")
print("Wind = API aggregate wind percentage, plus WIND_EMB if configured")
print("=" * 78)

for season in SEASON_ORDER_API:
    info = season_wind_summary.get(season)
    if not info:
        print(f"\n{season}: no data.")
        continue
    print(f"\n{season}  ({info['n_days']} days)")
    print("-" * 78)
    print(f"  Season average wind            : {info['season_avg_pct']:6.2f}%")
    print(f"    closest day                  : {info['rep_day'].date()}  ({info['rep_day_pct']:5.2f}%)")
    print(f"  Top {int(DECILE*100)}% wind days (n={info['top10_n']:>2})        : {info['top10_avg_pct']:6.2f}%")
    print(f"    closest day                  : {info['top10_rep_day'].date()}  ({info['top10_rep_day_pct']:5.2f}%)")
    print(f"  Bottom {int(DECILE*100)}% wind days (n={info['bottom10_n']:>2})     : {info['bottom10_avg_pct']:6.2f}%")
    print(f"    closest day                  : {info['bottom10_rep_day'].date()}  ({info['bottom10_rep_day_pct']:5.2f}%)")

print(f"\nRepresentative days for year_average LCA ({len(rep_days)} days):")
for r in rep_days:
    print(f"  {r['season']:<7} {r['kind']:<8} {r['date'].date()}  (~{r['target_pct']:5.2f}% wind)")

summary_df = pd.DataFrame({k: {kk: (vv.date() if hasattr(vv, 'date') else vv)
                               for kk, vv in v.items()}
                           for k, v in season_wind_summary.items()}).T
summary_df.index.name = "season"
summary_df
